In [ ]:
%matplotlib inline
import numpy as np
import h5py
import os
import re
import matplotlib.pyplot as plt
import CoolProp.CoolProp as CP
import pandas as pd
from ipywidgets import interactive
from keras.models import load_model,save_model
from sklearn.metrics import mean_squared_error,r2_score
from scipy.spatial.transform import Rotation as R
from scipy.ndimage import gaussian_filter1d
from scipy.signal import savgol_filter

# from pysr import PySRRegressor
from flowforge.materials.Fluid import User_Fluid,Hitec
print("Number of CPU cores available", os.cpu_count())


In [ ]:
def get_feature_indices(dataset_labels, model_labels):
    """Finds the indices of model features in the dataset feature list."""
    indices = [np.where(dataset_labels == label)[0][0] for label in model_labels if label in dataset_labels]

    if len(indices) != len(model_labels):
        missing = [label for label in model_labels if label not in dataset_labels]
        print(f"Warning: Missing features in dataset: {missing}")

    return indices

def fluctuation_adjusted_error(y_true, y_pred):
    """
    Computes Fluctuation-Adjusted Error (FAE).
    """
    std_true = np.std(y_true, axis=0)  # Standard deviation of true values
    return np.abs(y_pred - y_true) / std_true

def mean_absolute_percentage_error(y_true, y_pred):
    """
    Computes Mean Absolute Percentage Error (MAPE)

    Args:
        y_true (array-like): True values
        y_pred (array-like): Predicted values

    Returns:
        float: MAPE value in percentage
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

import numpy as np

def relative_root_mean_squared_error(y_true, y_pred):
    """
    Computes the Relative Root Mean Squared Error (RRMSE).

    RRMSE = RMSE / mean(y_true)

    Parameters:
    - y_true: np.array, true values
    - y_pred: np.array, predicted values

    Returns:
    - rrmse: float, relative root mean squared error
    """
    mse = np.mean((y_pred - y_true) ** 2)
    rmse = np.sqrt(mse)
    mean_y_true = np.mean(y_true)

    if mean_y_true == 0:
        return np.nan  # Avoid division by zero

    rrmse = rmse / mean_y_true
    return rrmse

def area_average(F,axis,dA):
    return np.average(F,axis,dA)

def weighted_mean(F_k,alpha_k,dA,axis = 0):
    alphaF_k_avg = area_average(alpha_k*F_k,axis,dA)
    alpha_k_avg = area_average(alpha_k,axis,dA)

    alpha_k_avg_safe = np.where(np.abs(alpha_k_avg) < 1e-12, np.nan, alpha_k_avg)

    alpha_F_k_weighted = alphaF_k_avg/alpha_k_avg_safe

    return np.where(np.isnan(alpha_F_k_weighted), 0, alpha_F_k_weighted)

def rho_mixture_avg(rho_l,rho_g,alpha_g_avg):
    return alpha_g_avg*rho_g + ((1-alpha_g_avg)*rho_l)

def velocity_mean(rho_l,rho_g,alpha_g,rho_m_avg,v_l_mean,v_g_mean):
    return (alpha_g*rho_g*v_g_mean + ((1-alpha_g)*rho_l*v_l_mean))/rho_m_avg

def Re(rho,v,D_h,mu):
    return (rho*v*D_h)/mu

def Pr(c_p,mu,kappa):
    return (c_p*mu)/kappa

def Eo(D_h, rho_gas, rho_liquid, gamma_liquid, g_z = 9.81):
    return (g_z*(D_h**2)*(rho_liquid-rho_gas))/gamma_liquid

def Mo(mu_liquid,rho_liquid,rho_gas,gamma_liquid,g_z = 9.81):
    return (g_z*(mu_liquid**4)*(rho_liquid-rho_gas))/((rho_liquid**2)*(gamma_liquid**3))

def We(rho_liquid,v,D_h,gamma_liquid):
    return (rho_liquid*(v**2)*D_h)/gamma_liquid

def Ga(mu_liquid,rho_liquid,gamma_liquid, g_z = 9.81):
    return (g_z*mu_liquid**4)/(rho_liquid*(gamma_liquid**3))

def volumetric_flux_closed(
    rho_m_avg,
    v_m_bar,
    alpha_d_avg,
    rho_c,
    rho_d,
    V_dj_w_avg,
    C_0,
    eps_denom=1e-14,
):
    """
    Computes the area-averaged volumetric flux, <j>, using the closed-form relation:

        <j> = ( <rho_m> * v_m_bar + <alpha_d> * (rho_c - rho_d) * <V_dj>_w )
              ------------------------------------------------------------
              ( <rho_m> - <alpha_d> * (rho_c - rho_d) * (C_0 - 1) )

    Parameter mapping to LaTeX:
      rho_m_avg   -> <rho_m>
      v_m_bar     -> \\overline{v_m}
      alpha_d_avg -> <alpha_d>
      rho_c       -> rho_c
      rho_d       -> rho_d
      V_dj_w_avg  -> <V_dj>_w  (your \\avgweighted{V_dj})
      C_0         -> C_0

    Notes:
      - Accepts floats or numpy arrays, broadcasts as needed.
      - eps_denom prevents division by (near) zero.
    """
    rho_m_avg   = np.asarray(rho_m_avg, dtype=float)
    v_m_bar     = np.asarray(v_m_bar, dtype=float)
    alpha_d_avg = np.asarray(alpha_d_avg, dtype=float)
    rho_c       = np.asarray(rho_c, dtype=float)
    rho_d       = np.asarray(rho_d, dtype=float)
    V_dj_w_avg  = np.asarray(V_dj_w_avg, dtype=float)
    C_0         = np.asarray(C_0, dtype=float)

    drho = (rho_c - rho_d)

    numerator = rho_m_avg * v_m_bar + alpha_d_avg * drho * V_dj_w_avg
    denominator = rho_m_avg - alpha_d_avg * drho * (C_0 - 1.0)

    denom_safe = np.where(np.abs(denominator) < eps_denom, np.nan, denominator)
    j_avg = numerator / denom_safe

    return np.where(np.isnan(j_avg), 0.0, j_avg)

def rotation_angle(theta = 0):
    # Define the gravity vector
    vec = np.array([0, 1, 0])

    rotation_matrix = R.from_euler('z', theta, degrees=True).as_matrix()

    # Apply the rotation
    rotated = (rotation_matrix @ vec)  # Matrix-vector multiplication

    rotated = np.where(np.abs(rotated) < 1e-10, 0, rotated)  # Small threshold
    return rotated

sqrt = np.sqrt
square = np.square

In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

# ============================================================
# Base directory resolution (JUPYTER SAFE)
# ============================================================

cwd = Path.cwd()

if (cwd / "data").exists():
    foam_base_dir = cwd
    data_root = foam_base_dir / "data"
else:
    if cwd.name == "data":
        foam_base_dir = cwd.parent
        data_root = cwd
    else:
        foam_base_dir = cwd.parent
        data_root = cwd

semiprocessed_root = data_root / "semiprocessed"
case_matrix_root   = data_root / "case_matrix"

print("foam_base_dir       :", foam_base_dir)
print("data_root           :", data_root)
print("semiprocessed_root  :", semiprocessed_root)
print("case_matrix_root    :", case_matrix_root)

# ============================================================
# Regex helpers (NEW NAMING)
# ============================================================

DATASET_CSV_RE = re.compile(r"^dataset_(\d+)(?:_|$)", re.IGNORECASE)
DATASET_DIR_RE = re.compile(r"^dataset_(\d+)$", re.IGNORECASE)
RUN_DIR_RE     = re.compile(r"^run_(\d+)$", re.IGNORECASE)
H5_RE          = re.compile(r"^sliced_points_(\d+(?:\.\d+)?)\.h5$", re.IGNORECASE)

# ============================================================
# Data structures
# ============================================================

data_entries = []

def extract_dataset_id_from_csv(csv_path: Path) -> int | None:
    """
    Extract dataset id from CSV stem:
      dataset_<id>_something.csv  -> <id>
      dataset_<id>.csv            -> <id> (rare but allowed)
    """
    m = DATASET_CSV_RE.match(csv_path.stem)
    if not m:
        return None
    return int(m.group(1))


def get_run_nums(semiproc_root: Path, material: str, dataset_name: str) -> list[int]:
    """
    Find run numbers under:
        data/semiprocessed/<material>/<dataset_name>/run_<n>
    """
    path = semiproc_root / material / dataset_name
    if not path.exists():
        print(f"[WARNING] Missing semiprocessed path: {path}")
        return []

    run_nums = []
    for d in path.iterdir():
        if not d.is_dir():
            continue
        m = RUN_DIR_RE.match(d.name)
        if m:
            run_nums.append(int(m.group(1)))

    return sorted(run_nums)


def find_end_times(semiproc_root: Path, df_conditions: pd.DataFrame) -> pd.Series:
    """
    For each row in df_conditions, look under:
        data/semiprocessed/<material>/dataset_<id>/run_<n>/
    for files like:
        sliced_points_<time>.h5
    and record the max time as 'end_time'.
    """
    end_times = []

    for _, row in df_conditions.iterrows():
        material = row["material"]
        dataset_name = row["dataset_name"]
        run = int(row["run"])

        run_dir = semiprocessed_root / material / dataset_name / f"run_{run}"

        if not run_dir.exists():
            end_times.append(np.nan)
            continue

        time_vals = []
        for f in run_dir.iterdir():
            if not f.is_file():
                continue
            m = H5_RE.match(f.name)
            if m:
                time_vals.append(float(m.group(1)))

        if not time_vals:
            end_times.append(np.nan)
            continue

        end_times.append(max(time_vals))

    return pd.Series(end_times, name="end_time")


def add_block_from_csv(material: str, dataset_id: int, dataset_name: str, run_nums: list[int], lhs_df: pd.DataFrame):
    """
    Build data_entries rows by mapping each run_<n> to the correct row in lhs_df.
    Uses 'case_id' if present; otherwise assumes run order matches CSV order (run-1 index).
    """
    has_case_id = "case_id" in lhs_df.columns

    for run in run_nums:
        if has_case_id:
            match = lhs_df.loc[lhs_df["case_id"] == run]
            if match.empty:
                print(f"[WARNING] No CSV row with case_id={run} for {material}/{dataset_name}. Skipping run_{run}.")
                continue
            row = match.iloc[0]
        else:
            idx = run - 1
            if idx < 0 or idx >= len(lhs_df):
                print(f"[WARNING] run_{run} is out of CSV row range for {material}/{dataset_name}. Skipping.")
                continue
            row = lhs_df.iloc[idx]

        entry = {
            "material": material,
            "dataset_id": dataset_id,
            "dataset_name": dataset_name,   # e.g., dataset_4
            "run": run,
            "T": float(row["T"]),
            "Angle": float(row["Angle"]),
            "U_gas_inlet": float(row["U_gas_inlet"]),
            "U_liquid": float(row["U_liquid"]),
            "centroid": float(row["centroid"]),
            "radius": float(row["radius"]),
        }
        data_entries.append(entry)


# ============================================================
# DATASET DEFINITIONS (REFRESHED FOR NEW LAYOUT)
# ============================================================
# Put your CSVs here (in data/case_parameter_matrix).
# You can optionally override dataset_id if a CSV doesn't start with dataset_<id>_...

DATASETS = [
    # {"csv": "dataset_30_vertical_upward_12_samples.csv", "material": "hitec_argon"},
    # {"csv": "dataset_70_variable_flow_12_samples.csv",   "material": "hitec_argon"},
    # {"csv": "dataset_1600_variable_flow_40_samples.csv", "material": "hitec_argon"},
    # {"csv": "dataset_1700_vertical_downward_20_samples.csv", "material": "hitec_argon"},
    # {"csv": "dataset_1800_vertical_downward_18_samples.csv", "material": "hitec_argon"},
    {"csv": "dataset_1_vertical_upward_4_samples.csv", "material": "hitec_argon"},
    {"csv": "dataset_2_vertical_upward_12_samples.csv", "material": "hitec_argon"},
]

# ============================================================
# Build df_conditions for all datasets that have semiprocessed data
# ============================================================

data_entries = []  # reset in case cell is rerun

for cfg in DATASETS:
    csv_path = case_matrix_root / cfg["csv"]
    if not csv_path.exists():
        print(f"[WARNING] Case-matrix CSV not found: {csv_path}")
        continue

    lhs_df = pd.read_csv(csv_path)

    required_cols = ["T", "U_gas_inlet", "U_liquid", "Angle", "centroid", "radius"]
    missing = [c for c in required_cols if c not in lhs_df.columns]
    if missing:
        print(f"[WARNING] Missing columns in {csv_path.name}: {missing}. Skipping.")
        continue

    material = cfg["material"]

    # Infer dataset id from CSV name unless explicitly overridden
    dataset_id = cfg.get("dataset_id", None)
    if dataset_id is None:
        dataset_id = extract_dataset_id_from_csv(csv_path)
    if dataset_id is None:
        print(f"[WARNING] Could not infer dataset id from CSV name: {csv_path.name}. Add cfg['dataset_id']. Skipping.")
        continue

    dataset_name = f"dataset_{dataset_id}"

    # Runs under data/semiprocessed/<material>/dataset_<id>/run_<n>
    run_nums = get_run_nums(semiprocessed_root, material, dataset_name)
    if not run_nums:
        print(f"[INFO] No run_<n> folders yet under {material}/{dataset_name}")
        continue

    add_block_from_csv(material, dataset_id, dataset_name, run_nums, lhs_df)

# Make DataFrame for all datasets
df_conditions = pd.DataFrame(data_entries)

if df_conditions.empty:
    print("df_conditions is empty (no datasets with matching semiprocessed data).")
else:
    df_conditions["end_time"] = find_end_times(semiprocessed_root, df_conditions)
    df_conditions = df_conditions.dropna(subset=["end_time"]).reset_index(drop=True)

    print("=== df_conditions (all datasets with data) ===")
    display(df_conditions)

    # ============================================================
    # Lists for EVERY parameter (match your CSV columns + metadata)
    # ============================================================
    material_list      = np.array(df_conditions["material"])
    dataset_list       = np.array(df_conditions["dataset_id"])      # dataset_num in your zip
    dataset_name_list  = np.array(df_conditions["dataset_name"])    # optional but useful
    run_num_list       = np.array(df_conditions["run"])

    T_list             = np.array(df_conditions["T"])
    angle_num_list     = np.array(df_conditions["Angle"])
    U_g_inlet          = np.array(df_conditions["U_gas_inlet"])     # U_g_num in your zip
    U_l_list           = np.array(df_conditions["U_liquid"])
    centroid_list      = np.array(df_conditions["centroid"])
    radius_list        = np.array(df_conditions["radius"])

    end_time_list      = np.array(df_conditions["end_time"])

    print("\nCounts:")
    print("  total rows         :", len(df_conditions))
    print("  unique materials   :", df_conditions["material"].nunique())
    print("  unique datasets    :", df_conditions["dataset_name"].nunique())

# ============================================================
# Example zip loop (your current one will still work)
# ============================================================
# for material, dataset_num, run_num, angle_num, U_g_num, end_time in zip(
#     material_list,
#     dataset_list,
#     run_num_list,
#     angle_num_list,
#     U_g_inlet,
#     end_time_list
# ):
#     print(material, dataset_num, run_num, angle_num, U_g_num, end_time)
#
# If you want ALL parameters in the loop:
# for material, dataset_num, run_num, T, angle, Ug, Ul, cen, rad, end_time in zip(
#     material_list, dataset_list, run_num_list,
#     T_list, angle_num_list, U_g_inlet, U_l_list, centroid_list, radius_list, end_time_list
# ):
#     print(material, dataset_num, run_num, T, angle, Ug, Ul, cen, rad, end_time)


In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import h5py

# ============================================================
# Base directory resolution (JUPYTER SAFE)
# ============================================================

cwd = Path.cwd()

if (cwd / "data").exists():
    foam_base_dir = cwd
    data_root = foam_base_dir / "data"
else:
    if cwd.name == "data":
        foam_base_dir = cwd.parent
        data_root = cwd
    else:
        foam_base_dir = cwd.parent
        data_root = cwd

semiprocessed_root = data_root / "semiprocessed"
samples_root       = data_root / "samples"

print("foam_base_dir       :", foam_base_dir)
print("data_root           :", data_root)
print("semiprocessed_root  :", semiprocessed_root)
print("samples_root        :", samples_root)

# ============================================================
# Helpers
# ============================================================

def get_sorted_time_values(run_dir, prefix="sliced_points"):
    """
    Return sorted list of time values for a given run_dir based on
    filenames like '<prefix>_3.5.h5' (e.g., sliced_points_3.5.h5).
    """
    run_dir = Path(run_dir)
    if not run_dir.exists():
        return []

    h5_files = [
        f.name for f in run_dir.iterdir()
        if f.is_file() and f.suffix == ".h5" and f.name.startswith(prefix + "_")
    ]

    time_vals = []
    for fname in h5_files:
        m = re.match(rf"^{re.escape(prefix)}_(\d+(?:\.\d+)?)\.h5$", fname)
        if m:
            time_vals.append(float(m.group(1)))

    return sorted(time_vals)

# ============================================================
# Your refactor: ONLY the path logic is changed
# ============================================================
#
# New layout assumed:
#   data/semiprocessed/<material>/dataset_<dataset_id>/run_<run_id>/sliced_points_<t>.h5
#
# Runs are now: run_1, run_2, ...
# Datasets are now: dataset_1, dataset_2, ...
#
# Everything else below is kept the same.

# If your notebook defines these elsewhere, you can remove these placeholders:
# preprocessed = False
# data_folder = str(semiprocessed_root)  # base folder for semiprocessed
# prefix = "sliced_points"
#
# You also need these arrays created earlier in your notebook:
# material_list, dataset_list, run_num_list, angle_num_list, U_g_inlet, end_time_list

# preprocessed = True
preprocessed = False

if not preprocessed:
    data_from_folder = f"{os.getcwd()}/data/semiprocessed"
    prefix = "sliced_points"

    initial1 = True
    for material, dataset_num, run_num, angle_num, U_g_num, end_time in zip(
        material_list,
        dataset_list,
        run_num_list,
        angle_num_list,
        U_g_inlet,
        end_time_list
    ):
        initial2 = True
        print(
            "material", material,
            "dataset", dataset_num,
            "run", run_num,
            "Angle", angle_num,
            "U_g_inlet", U_g_num,
            "end_time", end_time
        )

        # =========================
        # PATH CHANGE (NEW LAYOUT)
        # =========================
        run_dir = os.path.join(data_from_folder, material, f"dataset_{dataset_num}", f"run_{run_num}")

        time_list = get_sorted_time_values(run_dir, prefix=prefix)

        half_index = len(time_list) // 2
        filtered_time_list = time_list[half_index:]
        print(filtered_time_list)

        for t in filtered_time_list:
            t = round(t, 3)
            y_len = 0  # position start length

            # =========================
            # PATH + FILENAME CHANGE
            # =========================
            h5_file = f"{prefix}_{t}.h5"
            h5_path = os.path.join(run_dir, h5_file)

            with h5py.File(h5_path, "r") as hf:
                labels = [item.decode("utf-8") if isinstance(item, bytes) else item for item in np.array(hf["labels"][:])]
                points = np.array(hf["points"][:])[y_len:]
                parameters_t = np.array(hf["parameters"][:])[y_len:]

            _, Y, _ = points.T[0], points.T[1], points.T[2]
            alpha_g = parameters_t.T[12]

            axial_position = np.average(points.T[1], axis=0)
            T_liquid       = np.average(parameters_t.T[2], axis=0)
            T_gas          = np.average(parameters_t.T[1], axis=0)
            alpha_gas      = np.average(parameters_t.T[12], axis=0)
            P              = np.average(parameters_t.T[-1], axis=0)

            if material == "hitec_argon":
                liquid = Hitec("Hitec")
                gas = User_Fluid(
                    name="Argon",
                    therm_cond_funct=lambda h: CP.PropsSI("L", "H", h, "P", P, "Argon"),
                    dens_funct=lambda h:       CP.PropsSI("D", "H", h, "P", P, "Argon"),
                    visco_funct=lambda h:      CP.PropsSI("V", "H", h, "P", P, "Argon"),
                    spec_heat_funct=lambda h:  CP.PropsSI("C", "H", h, "P", P, "Argon"),
                    temp_funct=lambda h:       CP.PropsSI("T", "H", h, "P", P, "Argon"),
                    entha_funct=lambda T:      CP.PropsSI("H", "T", T, "P", P, "Argon"),
                )
            elif material == "hitec_helium":
                liquid = Hitec("Hitec")
                gas = User_Fluid(
                    name="Helium",
                    therm_cond_funct=lambda h: CP.PropsSI("L", "H", h, "P", P, "Helium"),
                    dens_funct=lambda h:       CP.PropsSI("D", "H", h, "P", P, "Helium"),
                    visco_funct=lambda h:      CP.PropsSI("V", "H", h, "P", P, "Helium"),
                    spec_heat_funct=lambda h:  CP.PropsSI("C", "H", h, "P", P, "Helium"),
                    temp_funct=lambda h:       CP.PropsSI("T", "H", h, "P", P, "Helium"),
                    entha_funct=lambda T:      CP.PropsSI("H", "T", T, "P", P, "Helium"),
                )
            elif material == "water_air":
                liquid = User_Fluid(
                    name="Water",
                    therm_cond_funct=lambda h: CP.PropsSI("L", "H", h, "P", P, "Water"),
                    dens_funct=lambda h:       CP.PropsSI("D", "H", h, "P", P, "Water"),
                    visco_funct=lambda h:      CP.PropsSI("V", "H", h, "P", P, "Water"),
                    surf_tens_funct=lambda h:  CP.PropsSI("I", "Q", 0, "T", CP.PropsSI("T", "H", h, "P", P, "Water"), "Water"),
                    spec_heat_funct=lambda h:  CP.PropsSI("C", "H", h, "P", P, "Water"),
                    temp_funct=lambda h:       CP.PropsSI("T", "H", h, "P", P, "Water"),
                    entha_funct=lambda T:      CP.PropsSI("H", "T", T, "P", P, "Water"),
                )
                gas = User_Fluid(
                    name="Air",
                    therm_cond_funct=lambda h: CP.PropsSI("L", "H", h, "P", P, "Air"),
                    dens_funct=lambda h:       CP.PropsSI("D", "H", h, "P", P, "Air"),
                    visco_funct=lambda h:      CP.PropsSI("V", "H", h, "P", P, "Air"),
                    spec_heat_funct=lambda h:  CP.PropsSI("C", "H", h, "P", P, "Air"),
                    temp_funct=lambda h:       CP.PropsSI("T", "H", h, "P", P, "Air"),
                    entha_funct=lambda T:      CP.PropsSI("H", "T", T, "P", P, "Air"),
                )
            else:
                raise Exception("Unknown liquid-gas mixture name:", material)

            # --- Thermophysical Property Evaluation ---
            h_liquid = liquid.enthalpy(T_liquid)
            h_gas    = gas.enthalpy(T_gas)

            mu_liquid = liquid.viscosity(h_liquid)
            mu_gas    = gas.viscosity(h_gas)

            rho_liquid = liquid.density(h_liquid)
            rho_gas    = gas.density(h_gas)

            cp_liquid = liquid.specific_heat(h_liquid)
            cp_gas    = gas.specific_heat(h_gas)

            k_liquid = liquid.conductivity(h_liquid)
            k_gas    = gas.conductivity(h_gas)

            sigma_liquid = liquid.surface_tension(h_liquid)

            # --- Geometric Properties and Angles ---
            theta      = np.ones(len(axial_position)) * angle_num
            angle_cos  = np.round(np.cos(np.deg2rad(theta)), 6)
            angle_sin  = np.round(np.sin(np.deg2rad(theta)), 6)
            g_z        = 9.81

            dA          = parameters_t.T[0]                  # Area array
            total_area  = dA.T[0].sum()
            D_h         = ((total_area / np.pi) ** 0.5 * 2) * np.ones(len(axial_position))

            P_avg = area_average(parameters_t.T[-1], axis=0, dA=dA)

            # --- Velocities and Void Fraction Calculations ---
            v_l = parameters_t.T[10, :, :]
            v_g = parameters_t.T[7, :, :]

            j_l = (1 - alpha_g) * v_l
            j_g = alpha_g * v_g
            j   = j_l + j_g

            V_dj_local = v_g - j

            v_l_mean = weighted_mean(v_l, 1 - alpha_g, dA, axis=0)
            v_g_mean = weighted_mean(v_g, alpha_g, dA, axis=0)

            # --- Averaged Properties ---
            alpha_g_avg = area_average(alpha_g, axis=0, dA=dA)
            rho_m_avg   = rho_mixture_avg(rho_liquid, rho_gas, alpha_g_avg)

            j_avg_CFD   = alpha_g_avg * v_g_mean + (1 - alpha_g_avg) * v_l_mean
            j_g_avg     = alpha_g_avg * v_g_mean
            j_l_avg     = (1 - alpha_g_avg) * v_l_mean

            v_m_mean = velocity_mean(rho_liquid, rho_gas, alpha_g_avg, rho_m_avg, v_l_mean, v_g_mean)

            m_dot_l_mean = rho_m_avg * v_l_mean * total_area
            m_dot_g_mean = j_g_avg * rho_gas * total_area
            m_dot_m_mean = rho_liquid * v_m_mean * total_area

            # --- Dimensionless Numbers ---
            Re_l = Re(rho_liquid, v_m_mean, D_h, mu_liquid)
            Pr_l = Pr(cp_liquid, mu_liquid, k_liquid)
            We_l = We(rho_liquid, v_m_mean, D_h, sigma_liquid)

            # --- Normalized Velocity Quantities ---
            j_star = j_avg_CFD / (np.sqrt(2) * ((g_z * sigma_liquid * (rho_liquid - rho_gas)) / (rho_liquid ** 2)) ** 0.25)
            j_plus = j_avg_CFD / (((g_z * sigma_liquid * (rho_liquid - rho_gas)) / (rho_liquid ** 2)) ** 0.25)

            # --- CFD-Based Constitutive Parameters ---
            C_0_CFD = weighted_mean(j_l + j_g, alpha_g, dA=dA, axis=0) / j_avg_CFD

            V_dj_local_mean_CFD      = weighted_mean(V_dj_local, alpha_g, dA=dA, axis=0)
            V_dj_local_mean_plus_CFD = V_dj_local_mean_CFD / (
                np.sqrt(2) * ((g_z * sigma_liquid * (rho_liquid - rho_gas)) / (rho_liquid ** 2)) ** 0.25
            )

            V_dj_mean_CFD = v_g_mean - j_avg_CFD

            # --- Ishii-Based Correlations ---
            C_0_Ishii = 1.2 - 0.2 * (rho_gas / rho_liquid) ** 2
            V_dj_local_mean_Ishii = np.sqrt(2) * ((g_z * sigma_liquid * (rho_liquid - rho_gas)) / (rho_liquid ** 2)) ** 0.25

            constant_Ishii  = ((g_z * sigma_liquid * (rho_liquid - rho_gas)) / (rho_liquid ** 2)) ** 0.25
            constant_Ishii2 = np.sqrt((g_z * D_h * (rho_liquid - rho_gas)) / rho_liquid)

            # --- Sanity Check: Drift Velocity Mean ---
            V_dj_mean_check = (1 - alpha_g_avg) * (v_g_mean - v_l_mean)
            assert np.allclose(V_dj_mean_CFD, V_dj_mean_check, rtol=1e-3), "Mismatch in V_dj_mean_CFD vs alt form"

            # --- Sanity Check: Inverse Reynolds Number ---
            inv_Re_l = 1 / Re_l
            Re_l_check = rho_liquid * v_m_mean * D_h / mu_liquid
            assert np.allclose(Re_l, Re_l_check, rtol=1e-3), "Mismatch in Re_l vs recalculated form"
            assert np.allclose(inv_Re_l, mu_liquid / (rho_liquid * v_m_mean * D_h), rtol=1e-3), "inv_Re_l does not match reciprocal form"

            # --- Sanity Check: Inverse Prandtl Number ---
            inv_Pr_l = 1 / Pr_l
            Pr_l_check = cp_liquid * mu_liquid / k_liquid
            assert np.allclose(Pr_l, Pr_l_check, rtol=1e-3), "Mismatch in Pr_l vs recalculated form"
            assert np.allclose(inv_Pr_l, k_liquid / (cp_liquid * mu_liquid), rtol=1e-3), "inv_Pr_l does not match reciprocal form"

            # --- Sanity Check: Inverse Weber Number ---
            inv_We_l = 1 / We_l
            We_l_check = rho_liquid * v_m_mean ** 2 * D_h / sigma_liquid
            assert np.allclose(We_l, We_l_check, rtol=1e-3), "Mismatch in We_l vs recalculated form"
            assert np.allclose(inv_We_l, sigma_liquid / (rho_liquid * v_m_mean ** 2 * D_h), rtol=1e-3), "inv_We_l does not match reciprocal form"

            # --- Sanity Check: Drift velocity V_dj_local ---
            V_dj_alt = (1 - alpha_g) * (v_g - v_l)
            assert np.allclose(V_dj_local, V_dj_alt, rtol=1e-3), "Mismatch in V_dj_local vs alternative form"

            # --- Sanity Check: Drift velocity V_dj_local ---
            j_avg_derived = volumetric_flux_closed(rho_m_avg, v_m_mean, alpha_g_avg, rho_liquid, rho_gas, V_dj_local_mean_CFD, C_0_CFD)
            assert np.allclose(j_avg_CFD, j_avg_derived, rtol=1e-3), "Mismatch in V_dj_local vs alternative form"

            parameter_values = np.array([
                # --- 1. Dimensionless Groups (and Inverse Forms) ---
                Re_l,
                Pr_l,
                We_l,
                1 / Re_l,
                1 / Pr_l,
                1 / We_l,
                # --- 2. Thermophysical Properties ---
                rho_liquid,
                cp_liquid,
                k_liquid,
                alpha_g_avg * rho_gas,
                (rho_gas / rho_liquid),
                1 / (rho_gas / rho_liquid),
                np.sqrt(rho_gas / rho_liquid),
                1 / np.sqrt(rho_gas / rho_liquid),
                (mu_gas / mu_liquid),
                1 / (mu_gas / mu_liquid),

                (mu_gas / rho_gas) / (mu_liquid / rho_liquid),
                (mu_liquid / rho_liquid) / (mu_gas / rho_gas),

                mu_liquid / np.sqrt(rho_liquid * sigma_liquid * np.sqrt(sigma_liquid / (9.81 * (rho_liquid - rho_gas)))),
                # --- 3. Hydrodynamic Properties ---
                v_m_mean / constant_Ishii,
                v_m_mean / constant_Ishii2,

                constant_Ishii / v_m_mean,
                constant_Ishii2 / v_m_mean,

                # --- 4. Geometric/Angle Inputs ---
                np.sqrt(sigma_liquid / (9.81 * (rho_liquid - rho_gas))) / D_h,
                D_h / np.sqrt(sigma_liquid / (9.81 * (rho_liquid - rho_gas))),
                0.35 * angle_sin + 0.45 * angle_cos,
                angle_cos,

                # --- 5. Drift-Flux Constants and Correlations ---
                alpha_g_avg,
                1 - alpha_g_avg,
                j_g_avg,
                m_dot_g_mean,
                m_dot_l_mean,
                m_dot_m_mean,
                v_g_mean,
                v_l_mean,
                P_avg,

                V_dj_local_mean_Ishii,
                C_0_Ishii,
                v_m_mean,
                rho_gas,
                rho_m_avg,

                # --- 6. CFD-based Drift-Flux Quantities (Outputs) ---
                C_0_CFD,
                V_dj_local_mean_CFD,
                j_avg_CFD,
                V_dj_mean_CFD
            ])

            if initial2:
                parameter_data = np.array([parameter_values])
                initial2 = False
            else:
                parameter_data = np.append(parameter_data, [parameter_values], axis=0)

        print(parameter_data.shape)

        if initial1:
            parameter_data_list = np.array([np.average(parameter_data, axis=0)])
            parameter_data_stat_list = np.array([np.std(parameter_data, axis=0)])
            T_l = np.array([T_liquid])

            Theta = np.array([theta])
            initial1 = False
        else:
            parameter_data_list = np.append(parameter_data_list, np.array([np.average(parameter_data, axis=0)]), axis=0)
            parameter_data_stat_list = np.append(parameter_data_stat_list, np.array([np.std(parameter_data, axis=0)]), axis=0)
            T_l = np.append(T_l, [T_liquid], axis=0)
            Theta = np.append(Theta, [theta], axis=0)

    print("")
    print(parameter_data_list.shape)
else:
    pass


In [ ]:
parameter_labels = np.array([
    # --- 1. Dimensionless Groups (and Inverse Forms) ---
    "Re_l",
    "Pr_l",
    "We_l",
    "inv_Re_l",
    "inv_Pr_l",
    "inv_We_l",

    # --- 2. Thermophysical Properties ---
    "rho_liquid",
    "C_p_liquid",
    "kappa_liquid",
    "gas_mass",
    "rho_gas/rho_liquid",
    "1/(rho_gas/rho_liquid)",
    "sqrt(rho_gas/rho_liquid)",
    "1/sqrt(rho_gas/rho_liquid)",
    "mu_gas/mu_liquid",
    "1/(mu_gas/mu_liquid)",
    "nu_gas/nu_liquid",
    "nu_liquid/nu_gas",
    "mu_liquid/sqrt_term",

    # --- 3. Hydrodynamic Properties ---
    "v_m_mean/correlation1",
    "v_m_mean/correlation2",

    "correlation1/v_m_mean",
    "correlation2/v_m_mean",

    # --- 4. Geometric/Angle Inputs ---
    "sqrt_term/D_h",
    "D_h/sqrt_term",
    "0.35*sin(theta)+0.45*cos(theta)",
    "cos(theta)",

    # --- 5. Drift-Flux Constants and Correlations ---
    "alpha_g_avg",
    "1 - alpha_g_avg",
    "j_g_avg",
    "m_dot_g_avg",
    "m_dot_l_avg",
    "m_dot_m_avg",
    "v_g_mean",
    "v_l_mean",
    "P_avg",

    "V_dj_local_mean_Ishii",
    "C_0_Ishii",
    "v_m_mean",
    "rho_g_avg",
    "rho_m_avg",


    # --- 6. CFD-based Drift-Flux Quantities (Outputs) ---
    "C_0_CFD",
    "V_dj_local_mean_CFD",
    "j_avg_CFD",
    "V_dj_mean_CFD"
])


parameter_titles = np.array([
    # --- 1. Dimensionless Groups ---
    r"$Re_l$",
    r"$Pr_l$",
    r"$We_l$",
    r"$Re_l^{-1}$",
    r"$Pr_l^{-1}$",
    r"$We_l^{-1}$",

    # --- 2. Thermophysical Properties ---
    r"Density of Liquid",
    r"Specific Heat of Liquid",
    r"Thermal Conductivity of Liquid",
    r"Gas Mass (m_{gas})",
    r"Density Ratio $\rho_g / \rho_l$",
    r"Inverse Density Ratio $(\rho_g / \rho_l)^{-1}$",
    r"Square Root Density Ratio $\sqrt{\rho_g / \rho_l}$",
    r"Inverse Square Root Density Ratio $1/\sqrt{\rho_g / \rho_l}$",
    r"Viscosity Ratio $\mu_g / \mu_l$",
    r"Inverse Viscosity Ratio $(\mu_g / \mu_l)^{-1}$",
    r"Kinematic Viscosity Ratio $\nu_g / \nu_l$",
    r"Kinematic Viscosity Ratio Inverse $\nu_l / \nu_g$",
    r"Morton-like Term $\mu_l / \sqrt{\sigma/(g(\rho_l - \rho_g))}$",

    # --- 3. Hydrodynamic Properties ---
    r"$v_m / C_1$",
    r"$v_m / C_2$",
    r"$C_1 / v_m$",
    r"$C_2 / v_m$",

    # --- 4. Geometric/Angle Inputs ---
    r"$\sqrt{\sigma / g(\rho_l - \rho_g)} / D_h$",
    r"$D_h / \sqrt{\sigma / g(\rho_l - \rho_g)}$",
    r"Weighted Inclination $0.35 \sin(\theta) + 0.45 \cos(\theta)$",
    r"Inclination $\cos(\theta)$",

    # --- 5. Drift-Flux Constants ---
    r"Void Fraction $\langle \alpha_g \rangle$",
    r"$1 - \langle \alpha_g \rangle$",
    r"$<j_g>$",
    r"$m_dot_g$",
    r"$m_{dot}^l$",
    r"$m_{dot}^m$",
    r"$<<v_g>>$",
    r"$<<v_l>>$",
    r"$<P>$",

    r"$V_dj_local$ (Ishii)",
    r"$C_0$ (Ishii)",
    r"$\overline{v_m}$",
    r"\rho_g$",
    r"\langle \rho_m \rangle$",


    # --- 6. CFD-based Drift-Flux Quantities ---
    r"Distribution Parameter $C_0$",
    r"Mean Local Drift Velocity $\ll V_{dj} \gg$ [m/s]",
    r"Area-averaged Volumetric Flux $<j>$ [m/s]",
    r"Mean Drift Velocity $\overline{V_{dj}}$ [m/s]"
])

parameter_labels.shape,parameter_titles.shape


In [ ]:
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
import numpy as np

def plot_multi_run_single_axis(
    dataset_num,
    run_list,
    plot_label="V_dj_local_mean_CFD",
    y_start=30,
    y_len=210,
    title="Time-averaged CFD Results",
    figsize=(6, 8),
    legend_loc="default",
    save=False,
    sigma_smooth=0.5,
    band_alpha=0.20,
    line_width=2.0,
    use_line_styles=False,   # set True if you still want dash patterns
):
    """
    Plot multiple CFD runs on a single axis with:
      - mean lines in default Matplotlib colors
      - variance (±1 std) band in the same color (transparent)
    """

    # find which parameter index to use
    plot_num = np.where(parameter_labels == plot_label)[0][0]
    Y = axial_position[y_start:y_len]

    # Optional line styles (only used if use_line_styles=True)
    line_styles = [
        "-", "--", ":", "-.",
        (0, (5, 2)),
        (0, (3, 5, 1, 5)),
        (0, (1, 1)),
        (0, (5, 1, 1, 1)),
    ]

    fig, ax = plt.subplots(figsize=figsize)

    for i, run in enumerate(run_list):
        # row in df_conditions corresponding to this dataset & run
        match = df_conditions[
            (df_conditions["run"] == run) &
            (df_conditions["dataset_id"] == dataset_num)
        ]
        if match.empty:
            continue
        row_idx = match.index[0]

        mat_temp  = round(df_conditions.loc[row_idx, "T"], 1)
        angle     = round(df_conditions.loc[row_idx, "Angle"], 1)
        U_g_inlet = round(df_conditions.loc[row_idx, "U_gas_inlet"], 3)
        U_l       = round(df_conditions.loc[row_idx, "U_liquid"], 3)

        # mean + std dev
        param_raw = parameter_data_list[row_idx, plot_num, y_start:y_len]
        param = gaussian_filter1d(param_raw, sigma=sigma_smooth) if sigma_smooth else param_raw
        param_std = parameter_data_stat_list[row_idx, plot_num, y_start:y_len]

        # label formatting (your special case)
        if dataset_num == 1700:
            label = (
                rf"$\theta={int(angle)}^\circ$, "
                rf"$T = {int(mat_temp)}\,\mathrm{{K}}$, "
                rf"$U_l = {U_l:.3f}$ m/s, "
                rf"$U_g^{{inlet}} = {U_g_inlet}$ m/s"
            )
        else:
            label = (
                rf"$\theta={int(angle)}^\circ$, "
                rf"$T = {int(mat_temp)}\,\mathrm{{K}}$, "
                rf"$U_l = {U_l}$ m/s, "
                rf"$U_g^{{inlet}} = {U_g_inlet}$ m/s"
            )

        linestyle = line_styles[i % len(line_styles)] if use_line_styles else "-"

        # Plot mean line FIRST; grab its auto-assigned color
        line, = ax.plot(param, Y, linestyle=linestyle, linewidth=line_width, label=label)
        color = line.get_color()

        # Plot variance band with same color
        ax.fill_betweenx(
            Y,
            param - param_std,
            param + param_std,
            color=color,
            alpha=band_alpha,
            linewidth=0
        )

    ax.set_title(title)
    ax.set_xlabel(parameter_titles[plot_num], fontsize=12)
    ax.set_ylabel("Distance from Inlet [m]", fontsize=12)
    ax.grid(True, linestyle="--", alpha=0.5)

    # Legend placement
    if legend_loc != "none":
        if legend_loc == "default":
            ax.legend(fontsize=11)
        else:
            ax.legend(loc=legend_loc, fontsize=11)

    fig.tight_layout()

    if save:
        run_str = "_".join(str(r) for r in run_list)
        fname = f"results/plots/CFD_{dataset_num}_{plot_label}_runs_{run_str}.png"
        fig.savefig(fname, dpi=500, bbox_inches="tight")
        print(f"Saved figure to {fname}")

    plt.show()


In [ ]:
from ipywidgets import (
    SelectMultiple, Dropdown, Button, HBox, VBox,
    Output, FloatText
)
from IPython.display import display, clear_output

# -----------------------------------------------------------
# Interactive widgets: pick dataset, runs, variable, figsize, legend
# -----------------------------------------------------------

# all datasets and runs from df_conditions
all_datasets = sorted(df_conditions["dataset_id"].unique())
default_dataset = all_datasets[0]

def get_runs_for_dataset(ds):
    return sorted(df_conditions[df_conditions["dataset_id"] == ds]["run"].unique())

# initial runs for default dataset
initial_runs = get_runs_for_dataset(default_dataset)

dataset_widget = Dropdown(
    options=all_datasets,
    value=default_dataset,
    description="Dataset:"
)

runs_widget = SelectMultiple(
    options=initial_runs,
    value=(initial_runs[0],) if initial_runs else (),
    description="Runs:",
    rows=8
)

plot_label_widget = Dropdown(
    options=list(parameter_labels),
    value="V_dj_local_mean_CFD" if "V_dj_local_mean_CFD" in parameter_labels else parameter_labels[0],
    description="Variable:"
)

# --- figure size controls ---
fig_width_widget = FloatText(
    value=6.0,
    description="Fig width:",
    step=0.5
)

fig_height_widget = FloatText(
    value=8.0,
    description="Fig height:",
    step=0.5
)

# --- legend position control ---
legend_widget = Dropdown(
    options=[
        ("Default", "default"),
        ("None", "none"),

        ("Top Left", "upper left"),
        ("Top Right", "upper right"),
        ("Top Center", "upper center"),

        ("Bottom Left", "lower left"),
        ("Bottom Right", "lower right"),
        ("Bottom Center", "lower center"),

        ("Center Left", "center left"),
        ("Center Right", "center right"),
        ("Center", "center"),
    ],
    value="default",
    description="Legend:"
)

save_button = Button(
    description="Save PNG (500 dpi)",
    button_style=""
)

# single output area for the plot
plot_output = Output()

def update_runs_options(change):
    """Update run options when dataset changes."""
    ds = change["new"]
    runs = get_runs_for_dataset(ds)
    runs_widget.options = runs
    if runs:
        runs_widget.value = (runs[0],)

dataset_widget.observe(update_runs_options, names="value")


def update_plot(*args):
    """Redraw the plot in a single output area."""
    if not runs_widget.value:
        with plot_output:
            clear_output(wait=True)
            print("Select at least one run.")
        return

    with plot_output:
        clear_output(wait=True)
        plot_multi_run_single_axis(
            dataset_num=dataset_widget.value,
            run_list=list(runs_widget.value),
            plot_label=plot_label_widget.value,
            figsize=(fig_width_widget.value, fig_height_widget.value),
            legend_loc=legend_widget.value,
            save=False
        )


def on_save_clicked(b):
    if not runs_widget.value:
        with plot_output:
            clear_output(wait=True)
            print("Select at least one run before saving.")
        return

    with plot_output:
        clear_output(wait=True)
        plot_multi_run_single_axis(
            dataset_num=dataset_widget.value,
            run_list=list(runs_widget.value),
            plot_label=plot_label_widget.value,
            figsize=(fig_width_widget.value, fig_height_widget.value),
            legend_loc=legend_widget.value,
            save=True
        )

# hook up widget changes to update_plot
runs_widget.observe(lambda change: update_plot(), names="value")
plot_label_widget.observe(lambda change: update_plot(), names="value")
dataset_widget.observe(lambda change: update_plot(), names="value")
fig_width_widget.observe(lambda change: update_plot(), names="value")
fig_height_widget.observe(lambda change: update_plot(), names="value")
legend_widget.observe(lambda change: update_plot(), names="value")
save_button.on_click(on_save_clicked)

# initial plot
update_plot()

# display controls + single plot area
display(
    VBox([
        HBox([dataset_widget, plot_label_widget]),
        HBox([runs_widget,
              VBox([fig_width_widget, fig_height_widget, legend_widget, save_button])]),
        plot_output
    ])
)